# 04 - Optimization Logic Deep Dive

**Dynamic Pricing Engine** - Price Optimization Mathematics

This notebook explains the core math behind the optimizer:

Revenue = Price * Demand(Price)

Demand(P) = D0 * (P / P0) ^ e

Where:
- D0 = demand at base price
- P0 = base/market price  
- e = price elasticity (negative)
- scipy.optimize finds P* that maximizes Revenue

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.models.optimizer import (
    demand_function, revenue_function, get_optimal_price, compute_revenue_curve
)
from src.utils.config import load_config

config = load_config()

## 1. Understanding Elasticity

In [ ]:
# Visualize how different elasticities affect the demand curve
prices = np.linspace(50, 300, 100)
base_price = 150
base_demand = 0.7

fig = go.Figure()
for e in [-0.5, -1.0, -1.5, -2.0, -2.5]:
    demands = [demand_function(p, base_demand, base_price, e) for p in prices]
    fig.add_trace(go.Scatter(x=prices, y=demands, name=f'e = {e}', mode='lines'))

fig.add_vline(x=base_price, line_dash='dash', line_color='gray')
fig.update_layout(
    title='Demand Curves at Different Elasticities',
    xaxis_title='Price ($)',
    yaxis_title='Demand (Occupancy Rate)',
    height=450,
)
fig.show()

## 2. Revenue Surfaces

In [ ]:
# Revenue curves for different elasticities
fig = go.Figure()
for e in [-0.5, -1.0, -1.5, -2.0]:
    revenues = [revenue_function(p, base_demand, base_price, e) for p in prices]
    opt = get_optimal_price(base_demand, base_price, e, floor_price=50, ceiling_price=300, config=config)
    
    fig.add_trace(go.Scatter(
        x=prices, y=revenues,
        name=f'e={e} (opt=${opt.optimal_price})',
        mode='lines',
    ))
    fig.add_trace(go.Scatter(
        x=[opt.optimal_price], y=[opt.expected_revenue],
        mode='markers', marker=dict(size=12, symbol='star'),
        showlegend=False,
    ))

fig.update_layout(
    title='Revenue Curves: Finding the Optimal Price',
    xaxis_title='Price ($)',
    yaxis_title='Revenue ($)',
    height=500,
)
fig.show()

## 3. Key Mathematical Insight

For the constant-elasticity demand model:

**When |e| < 1** (inelastic): Optimal price = ceiling (raise prices, demand barely drops)

**When |e| = 1** (unit elastic): Revenue is constant across all prices

**When |e| > 1** (elastic): There exists an interior optimum where marginal revenue = 0

The analytical solution for unbounded optimization is:
P* = P0 * (e / (e + 1)) when e < -1

Our scipy optimizer finds this numerically within the bounded range.

In [ ]:
# Verify analytical vs numerical solution
e = -1.5
analytical_optimal = base_price * (e / (e + 1))
numerical = get_optimal_price(base_demand, base_price, e, floor_price=1, ceiling_price=1000, config=config)

print(f'Elasticity: {e}')
print(f'Analytical optimal: ${analytical_optimal:.2f}')
print(f'Numerical optimal:  ${numerical.optimal_price:.2f}')
print(f'Match: {abs(analytical_optimal - numerical.optimal_price) < 1.0}')